# Introduction
This notebook mimics the chi-square feature selecdtion from assignment1 while using PySpark instead of MapReduce.

Process steps:
- Calculate term-category counts and category total counts
- Calculate chi-square scores for each category
- Select top k terms

## Imports

In [1]:
from pyspark.sql import SparkSession
import json
import re

## Global Variables

In [2]:
REVIEWS_DATA_PATH = "hdfs:///dic_shared/amazon-reviews/full/reviews_devset.json"
STOPWORDS_PATH = f"../data/stopwords.txt"
OUTPUT_FILE = f"../result/output_rdd.txt"
TOP_K = 75

## Environment Setup

In [3]:
with open(STOPWORDS_PATH, "r") as f:
	STOPWORDS = set(f.read().split())

In [4]:
spark = SparkSession.builder.master("local[*]").appName("AmazonChiSquareRDD").getOrCreate()
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 07:14:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Word Count

In [5]:
TOKEN_SPLIT_RE = r"[\s\d\(\)\[\]\{\}\.\!\?\,\;\:\+\=\-_\"\'\`\~\#\@\&\*\%€\$\§\\\/]+"

def parse_review(line):
	review = json.loads(line)
	tokens = re.split(TOKEN_SPLIT_RE, review.get("reviewText", "").lower())
	tokens = {t for t in tokens if t and t not in STOPWORDS and len(t) > 1}
	return review.get("category", ""), tokens

In [6]:
reviews = sc.textFile(REVIEWS_DATA_PATH).map(parse_review)

term_category_counts = (
	reviews.flatMap(lambda x: [((term, x[0]), 1) for term in x[1]] + [(('__TOTAL__', x[0]), 1)])
	.reduceByKey(lambda a, b: a + b)
	.cache()
)

# Chi-Square

In [7]:
def chi_square(counts):
	total_term = sum(counts.values())
	scores = {}
	for cat in categories:
		a = counts.get(cat, 0)
		b = total_term - a
		c = category_totals[cat] - a
		d = total_reviews - (a + b + c)
		den = (a + b) * (c + d) * (a + c) * (b + d)
		scores[cat] = total_reviews * ((a * d - b * c) ** 2) / den if den else 0.0
	return scores

In [8]:
category_totals = dict(
	term_category_counts
	.filter(lambda kv: kv[0][0] == '__TOTAL__')
	.map(lambda kv: (kv[0][1], kv[1]))
	.collect()
)

total_reviews = sum(category_totals.values())
categories = sorted(category_totals)

top_terms_per_category = (
	term_category_counts
	.filter(lambda kv: kv[0][0] != '__TOTAL__')
	.map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
	.groupByKey()
	.mapValues(lambda items: chi_square(dict(items)))
	.flatMap(lambda kv: [((cat, kv[0]), score) for cat, score in kv[1].items()])
	.map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
	.groupByKey()
	.mapValues(lambda items: sorted(items, key=lambda x: (-x[1], x[0]))[:TOP_K])
	.collectAsMap()
)

26/05/16 07:14:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

## Output

In [9]:
lines = []
vocabulary = set()
for cat in sorted(top_terms_per_category):
	terms = top_terms_per_category[cat]
	vocabulary.update(term for term, _ in terms)
	lines.append(f"{cat} " + " ".join(f"{term}:{round(score, 4)}" for term, score in terms))
lines.append(" ".join(sorted(vocabulary)))

with open(OUTPUT_FILE, "w") as f:
	f.write("\n".join(lines) + "\n")